# 03 - Hiperparámetros, Estimadores Base Heterogéneos y Sensibilidad a Outliers

## Curso de Machine Learning: Modelos de Clasificación

Una característica única de `AdaBoostClassifier` en Scikit-Learn es que **no está restringido a usar árboles de decisión**: cualquier clasificador que acepte pesos de muestra (`sample_weight`) puede actuar como estimador base (`estimator`). En este cuaderno analizaremos la sintonización conjunta del estimador base (`max_depth`), el compromiso entre `n_estimators` y `learning_rate`, y el impacto del ruido de etiquetas (*label noise*) sobre los pesos exponenciales.

---

### Contenidos
1. Efecto de la Complejidad del Árbol Base (`max_depth=1` vs `2` vs `4`)
2. AdaBoost con Estimadores Base No Basados en Árboles (`LogisticRegression`)
3. Análisis Experimental de Sensibilidad a *Outliers* y Ruido en las Etiquetas
4. Conclusiones y Recomendaciones de Regularización


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. Interacción entre la Profundidad del Estimador Base y `n_estimators`

- **`max_depth=1` (*Decision Stump*)**: No modela interacciones entre variables dentro de un mismo árbol; excelente cuando los efectos son aditivos.
- **`max_depth=2` o `3`**: Permite capturar interacciones de orden 2 o 3 entre características, acelerando la convergencia pero incrementando el riesgo de sobreajuste si `learning_rate` es alto.


In [ ]:
X_syn, y_syn = make_classification(
    n_samples=1200, n_features=20, n_informative=12, n_redundant=4, flip_y=0.03, random_state=42
)
X_tr, X_te, y_tr, y_te = train_test_split(X_syn, y_syn, test_size=0.25, random_state=42, stratify=y_syn)

profundidades = [1, 2, 3, 5]
plt.figure(figsize=(10, 4.8))

for d in profundidades:
    clf = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=d, random_state=42),
        n_estimators=150,
        learning_rate=0.2,
        random_state=42
    )
    clf.fit(X_tr, y_tr)
    scores_te = list(clf.staged_score(X_te, y_te))
    plt.plot(range(1, 151), scores_te, lw=2, label=f'Base DecisionTree (max_depth={d}) -> Final: {scores_te[-1]*100:.1f}%')

plt.xlabel('Numero de Estimadores (n_estimators)', fontweight='bold')
plt.ylabel('Exactitud en Test', fontweight='bold')
plt.title('Desempeño de AdaBoost segun la Profundidad del Arbol Base', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()


## 2. Sensibilidad al Ruido de Etiquetas (*Label Noise*) y Outliers

Dado que la función de pérdida exponencial asigna pesos $w_i \propto e^{-y_i F(\mathbf{x}_i)}$, las observaciones atípicas o con etiquetas mal anotadas (*label noise*) acumulan la mayor parte del peso total del entrenamiento, deteriorando la frontera de decisión. Verifiquemos este fenómeno inyectando niveles crecientes de ruido (`flip_y`) y comparando AdaBoost frente a Random Forest y Gradient Boosting:


In [ ]:
niveles_ruido = [0.0, 0.05, 0.10, 0.20, 0.30]
res_ruido = []

for noise in niveles_ruido:
    X_n, y_n = make_classification(
        n_samples=1000, n_features=15, n_informative=10, flip_y=noise, random_state=42
    )
    X_tr_n, X_te_n, y_tr_n, y_te_n = train_test_split(X_n, y_n, test_size=0.25, random_state=42, stratify=y_n)
    
    ada = AdaBoostClassifier(n_estimators=100, learning_rate=0.8, random_state=42).fit(X_tr_n, y_tr_n)
    gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42).fit(X_tr_n, y_tr_n)
    rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X_tr_n, y_tr_n)
    
    res_ruido.append({
        'Ruido en Etiquetas (%)': int(noise * 100),
        'AdaBoost (Exp Loss)': accuracy_score(y_te_n, ada.predict(X_te_n)) * 100,
        'Gradient Boosting (Log-Loss)': accuracy_score(y_te_n, gb.predict(X_te_n)) * 100,
        'Random Forest (Bagging)': accuracy_score(y_te_n, rf.predict(X_te_n)) * 100
    })

df_ruido = pd.DataFrame(res_ruido)
print(df_ruido.to_string(index=False))

df_ruido.set_index('Ruido en Etiquetas (%)').plot(marker='o', figsize=(9, 4.5), lw=2)
plt.ylabel('Exactitud en Test (%)', fontweight='bold')
plt.title('Degradacion ante Ruido en Etiquetas (Outliers): AdaBoost vs Otros Ensambles', fontweight='bold')
plt.tight_layout()
plt.show()
